# Step 6: Data Cleaning

**SageMaker Unified Studio Component**: Data Processing

**What you'll learn**: Handle missing data and prepare for ML

Note: In this sample use case, this step is innocuous because the dataset is already clean. In a real-world scenario, you would perform data cleaning here.

In [ ]:
import pandas as pd
import boto3
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

account_id = boto3.client('sts').get_caller_identity()['Account']
bucket_name = os.getenv('BUCKET_NAME', f'sagemaker-unified-overheat-demo-{account_id}')
print(f"Using bucket: {bucket_name}")

## Load Raw Data

In [ ]:
s3_path = f's3://{bucket_name}/data/raw/machines.csv'
df = pd.read_csv(s3_path)
print(f"Original rows: {len(df):,}")
df.head()

## Cleaning Operations

In [ ]:
# 1. Remove rows with missing temperature
df_clean = df.dropna(subset=['temperature', 'room_temp'])
print(f"Removed {len(df) - len(df_clean)} rows with missing values")

In [ ]:
# 2. Convert timestamp to datetime
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
print("Converted timestamp to datetime")

In [ ]:
# 3. Sort by timestamp
df_clean = df_clean.sort_values('timestamp').reset_index(drop=True)
print(f"Final rows: {len(df_clean):,}")

## Verify Cleaning

In [ ]:
# Check for nulls
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

# Check data types
print("\nData types:")
print(df_clean.dtypes)

## Save Cleaned Data

In [ ]:
# Save to Parquet (more efficient than CSV)
output_path = f's3://{bucket_name}/data/processed/clean_machines.parquet'
df_clean.to_parquet(output_path, index=False)
print(f"Saved to: {output_path}")

## Key Concepts

**Why Parquet?**
- Columnar format (faster for ML)
- Compressed (smaller storage)
- Preserves data types

**Next step**: Feature engineering